In [ ]:
%matplotlib widget

In [ ]:
import flammkuchen as fl
import numpy as np
from pathlib import Path
import tifffile as tiff
from split_dataset import SplitDataset

In [ ]:
import matplotlib.pyplot as plt
from fimpy.pipeline.roi_extraction import extract_traces

In [ ]:
from scipy.stats import zscore
from bouterin.plots.stimulus_log_plot import get_paint_function
from fimpylab.core.twop_experiment import TwoPExperiment
from motions.utilities import stim_vel_dir_dataframe, quantize_directions

In [ ]:
import colorspacious

In [ ]:
# make a color map

def JCh_to_RGB255(x):
    output = np.clip(colorspacious.cspace_convert(x, "JCh", "sRGB1"), 0, 1)
    return (output * 255).astype(np.uint8)

def color_stack(
        amp,
        angle,
        hueshift=2.5,
        amp_percentile=80,
        maxsat=50,
        lightness_min=100,
        lightness_delta=-40,
    ):
    output_lch = np.empty(amp.shape + (3,))
    maxamp = np.percentile(amp, amp_percentile)

    output_lch[:, :, 0] = (
            lightness_min + (np.clip(amp / maxamp, 0, 1)) * lightness_delta
    )
    output_lch[:, :, 1] = (np.clip(amp / maxamp, 0, 1)) * maxsat
    output_lch[:, :, 2] = (-angle + hueshift) * 180 / np.pi

    return JCh_to_RGB255(output_lch)

In [ ]:
master = Path(r"Z:\Hagar\e0075\e0075_v10 and e0040")
master = Path(r"Z:\Hagar\s11\e0075")

fish_list = list(master.glob("*_f*"))

master = Path(r"Z:\Hagar\s11\e0075")
master = Path(r"Z:\Hagar\e0075\v06 and v09")
fish_list = list(master.glob("*_f*"))


relevant = [f for f in fish_list if (f / "manual_rois_test_Sample.h5").exists()]
relevant




relevant

In [ ]:
###  extract traces for all fish
for fish in relevant:
    print(fish)
    try:
        if not (fish / "traces.h5").exists():
            print("Extracting traces")
            rois = fl.load(fish / 'manual_rois_test_Sample.h5', '/labels_rois')
            aligned = SplitDataset(fish / "aligned")[:,:,:,:]

            n_rois = np.max(rois)
            n_t = np.shape(aligned)[0]
            traces = np.zeros((n_rois, n_t))
            norm_traces = np.zeros((n_rois, n_t))

            t_imaging = np.arange(0, np.shape(aligned)[0]) // 3

            for roi in range(1, n_rois+1):
                curr_roi = np.where(rois == roi)
                #trace = aligned[:, i, curr_roi[1], curr_roi[2]]
                try:
                    z = curr_roi[0][1]
                    center_x = np.nanmean(curr_roi[2])
                    center_y = np.nanmean(curr_roi[1])

                    num_pix = curr_roi[1]
                    traces[roi-1] = np.nanmean(aligned[:, z, curr_roi[1], curr_roi[2]], axis=1)
                    norm_traces[roi-1] = zscore(traces[roi-1])

                except:
                    print("no rois")

            d = {'traces': traces,
                'norm_traces': norm_traces}
            fl.save(fish / 'traces.h5', d)
        else:
            print("Traces already extracted")
            
    except:
        print("Stupid fish")
    print('done')

In [ ]:
fish = relevant[-1]
fish

In [ ]:
num_trial = 10

In [ ]:
rois = fl.load(fish / 'manual_rois_test_Sample.h5', '/labels_rois')
tuning_map = fl.load(fish / 'tuning_map_new_90.h5')
#anatomy = tiff.imread(fish / 'anatomy.tif')

In [ ]:
try:
    traces = fl.load(fish / 'traces.h5', '/norm_traces')

except:    
    aligned = SplitDataset(fish / "aligned")[:,:,:,:]

In [ ]:
num_planes = np.shape(tuning_map)[0]
print("num planes: ", num_planes)

plane_list = list((fish / "suite2p").glob("*00*"))

In [ ]:
plane_ind = 5
plane = plane_list[plane_ind]
sensory_reg = fl.load(plane / 'sensory_regressors.h5')['regressors']
num_regressors = np.shape(sensory_reg)[0]
print(num_regressors)

In [ ]:
beh_path = list(plane.glob("*behavior*"))
beh_log = fl.load(beh_path[0])['data']
tail = beh_log['tail_sum']

In [ ]:
bin_centers, bins = quantize_directions([0], 8)

angles = np.stack(
        [
            np.full(len(bin_centers), 60),
            np.full(len(bin_centers), 60),
            (-bin_centers + 2.5) * 180 / np.pi,
        ],
        1)
s = JCh_to_RGB255(angles)
s = s/255


In [ ]:
color_list = ['lightgreen', 'turquoise', 'cyan', 'skyblue', 'mediumpurple', 'pink', 'orange', 'gold']
#color_list = ['lightgreen', 'turquoise', 'mediumpurple', 'pink']

In [ ]:
fig, ax = plt.subplots(2,2, figsize=(12,5), gridspec_kw={'width_ratios': [1,2], 'height_ratios': [1,4]})
for reg in range(num_regressors):
    
    ax[1, 0].imshow(tuning_map[plane_ind])
    ax[1, 0].invert_yaxis()
    
    curr_reg = sensory_reg[reg]
    t_start = np.where(np.diff(curr_reg) > 0)[0]
    
    t_end = np.where(np.diff(curr_reg) < 0)[0]
    print(t_start[0])
    print(t_end[0])
    
    for trial in range(num_trial):
        ax[1, 1].axvspan(
                t_start[trial],
                t_end[trial],
                facecolor=s[reg],
                alpha=0.5,
            )
        
    ax[0,0].axis('off')
    ax[0,1].axis('off')
    ax[0,1].plot(tail, c='black')

In [ ]:
fig.subplots_adjust(left=0.05, right=0.9)

In [ ]:
np.unique(rois)

In [ ]:
n_rois = np.max(rois)
n_t = np.shape(aligned)[0] 
t_imaging = np.arange(0, np.shape(aligned)[0]) 

traces = np.zeros((n_rois, n_t))
norm_traces = np.empty((n_rois, n_t))

for roi in range(1, n_rois+1):
    curr_roi = np.where(rois == roi)
    try:
        z = curr_roi[0][1]
        print(z)
        if z == plane_ind:
            center_x = np.nanmean(curr_roi[2])
            center_y = np.nanmean(curr_roi[1])
            ax[1, 0].scatter(center_x, center_y, s=2)

            num_pix = curr_roi[1]
            traces[roi-1] = np.nanmean(aligned[:, z, curr_roi[1], curr_roi[2]], axis=1)
            norm_traces[roi-1] = zscore(traces[roi-1])

            ax[1, 1].plot(t_imaging, norm_traces[roi-1] + (5 * (roi)), linewidth=0.5)
        else:
            print("no roi")
    except:
        print("no rois")

In [ ]:
file_name = "manual rois and traces " + str(plane_ind) + ".pdf"
fig.savefig(fish / file_name, dpi=300)
file_name = "manual rois and traces " + str(plane_ind) + ".jpg"
fig.savefig(fish / file_name, dpi=300)

In [ ]:
n_rois = np.max(rois)
n_t = np.shape(aligned)[0] 
t_imaging = np.arange(0, np.shape(aligned)[0]) 

traces = np.zeros((n_rois, n_t))
norm_traces = np.empty((n_rois, n_t))

for roi in range(1, n_rois+1):
    curr_roi = np.where(rois == roi)
    try:
        z = curr_roi[0][1]
        center_x = np.nanmean(curr_roi[2])
        center_y = np.nanmean(curr_roi[1])

        num_pix = curr_roi[1]
        traces[roi-1] = np.nanmean(aligned[:, z, curr_roi[1], curr_roi[2]], axis=1)
        norm_traces[roi-1] = zscore(traces[roi-1])
    except:
        print("no rois")

In [ ]:
n_rep = num_trial
n_options = 8
fs = 3
title_list = ['■□□□□□□□', '□■□□□□□□', '□□■□□□□□', '□□□■□□□□', '□□□□■□□□', '□□□□□■□□', '□□□□□□■□', '□□□□□□□■']

In [ ]:
ind_trace = 29
trace = norm_traces[ind_trace]


num_row = 2
num_col = 4
fig_regs3, ax_regs3 = plt.subplots(num_row, num_col, figsize=(8, 4), sharex=True, sharey=True)

for i in range(n_options):
    r = i // num_col
    c = np.mod(i, num_col)
    
    new_len = 40
    trial_trace = np.zeros((n_rep, new_len))
    
    reg_dif = np.diff(sensory_reg[i])
    t_start = np.where(reg_dif > 0)[0] - 10
    t_end = t_start + new_len
    t_vec = (np.arange(new_len) - 10 )/ fs
    
    for trial in range(n_rep):
        try:
            #rint(t_start[trial], t_end[trial])
            trial_trace[trial] = trace[t_start[trial]:t_end[trial]]
        except:
            print("stupid trace")
    
    trace_avg = np.nanmean(trial_trace, axis=0)
    trace_sem = np.nanstd(trial_trace, axis=0)/np.sqrt(n_rep)
    ax_regs3[r,c].plot(t_vec,  trace_avg, c='royalblue')
    ax_regs3[r,c].fill_between(t_vec, trace_avg - trace_sem, trace_avg + trace_sem, color='lightblue')
    ax_regs3[r,c].set_title(title_list[i])
    
    if i is not n_options//2:
        ax_regs3[r,c].axis('off')
        

        
        

In [ ]:
file_name = 'rf trace ' + str(ind_trace) + '.jpg'
fig_regs3.savefig(fish / file_name, dpi=300)

file_name = 'rf trace ' + str(ind_trace) + '.pdf'
fig_regs3.savefig(fish / file_name, dpi=300)

In [ ]:
fig, axs = plt.subplots(num_planes, 2, figsize=(12, 12), gridspec_kw={'width_ratios': [1,4]})
fig.subplots_adjust(top=0.99, bottom=0.1, left=0.1, right=0.9, wspace=0.1)
for i in range(num_planes):
    axs[i, 0].imshow(np.rot90(tuning_map[i], 0), vmin=0, vmax=1)
    axs[i, 0].axis('off')
    axs[i, 0].invert_yaxis()

    axs[i, 1].axis('off')
    
    for reg in range(num_regressors):
    
        axs[i, 0].imshow(tuning_map[i])
        axs[i, 0].invert_yaxis()

        curr_reg = sensory_reg[reg]
        t_start = np.where(np.diff(curr_reg) > 0)[0]

        t_end = np.where(np.diff(curr_reg) < 0)[0]

        for trial in range(num_trial):
            axs[i, 1].axvspan(
                    t_start[trial],
                    t_end[trial],
                    facecolor=s[reg],
                    alpha=0.5,
                )

    

In [ ]:
t_imaging = np.arange(0, np.shape(aligned)[0]) 

    
for roi in range(1, n_rois+1):
    curr_roi = np.where(rois == roi)
    #trace = aligned[:, i, curr_roi[1], curr_roi[2]]
    try:
        z = curr_roi[0][1]
        center_x = np.nanmean(curr_roi[2])
        center_y = np.nanmean(curr_roi[1])
        axs[z, 0].scatter(center_x, center_y, s=2)

        num_pix = curr_roi[1]
        traces[roi-1] = np.nanmean(aligned[:, z, curr_roi[1], curr_roi[2]], axis=1)
        norm_traces[roi-1] = zscore(traces[roi-1])

        axs[z, 1].plot(t_imaging, norm_traces[roi-1] + (5 * (roi)), linewidth=0.5)
    except:
        print("no rois")


    

In [ ]:
d = {'traces': traces,
    'norm_traces': norm_traces}
fl.save(fish / 'traces.h5', d)

In [ ]:
if nun_planes > 1:
    fig2, axs2 = plt.subplots(num_planes, 2, figsize=(8, 12), gridspec_kw={'width_ratios': [1,4]})
    fig2.subplots_adjust(top=0.99, bottom=0.1, left=0.1, right=0.9, wspace=0.1)
    for i in range(num_planes):
        axs2[i, 0].imshow(np.rot90(anatomy[i], 0), cmap="gray_r", vmin=0, vmax=1)
        axs2[i, 0].axis('off')
        axs2[i, 0].invert_yaxis()

        axs2[i, 1].axis('off')
        for stim in range(num_stim//3):
            axs2[i, 1].axvspan(
                t_values[stim, 0],
                t_values[stim, 1],
                facecolor=[stim_value[stim, 0], stim_value[stim, 1], stim_value[stim, 2]],
                alpha=0.5,
            )
else:
    fig2, axs2 = plt.subplots(num_planes, 2, figsize=(8, 4), gridspec_kw={'width_ratios': [1,4]})
    fig2.subplots_adjust(top=0.99, bottom=0.1, left=0.1, right=0.9, wspace=0.1)
    for i in range(num_planes):
        axs2[0].imshow(np.rot90(anatomy[i], 0), cmap="gray_r", vmin=0, vmax=1)
        axs2[0].axis('off')
        axs2[0].invert_yaxis()

        axs2[1].axis('off')
        for stim in range(num_stim//3):
            axs2[1].axvspan(
                t_values[stim, 0],
                t_values[stim, 1],
                facecolor=[stim_value[stim, 0], stim_value[stim, 1], stim_value[stim, 2]],
                alpha=0.5,
            )
    

In [ ]:
n_rois = np.max(rois)

t_imaging = np.arange(0, np.shape(aligned)[0]) // 3

if nun_planes > 1:    
    for roi in range(1, n_rois+1):
        curr_roi = np.where(rois == roi)
        #trace = aligned[:, i, curr_roi[1], curr_roi[2]]

        try:
            z = curr_roi[0][1]
            center_x = np.nanmean(curr_roi[2])
            center_y = np.nanmean(curr_roi[1])
            axs2[z, 0].scatter(center_x, center_y, s=2)

            num_pix = curr_roi[1]
            trace = zscore(np.nanmean(aligned[:, z, curr_roi[1], curr_roi[2]], axis=1))
            trace_reshape = np.reshape(trace[:768*3], (3, np.shape(aligned)[0]//3))
            trace_avg = np.nanmean(trace_reshape, axis=0)

            axs2[z, 1].plot(t_imaging[:768], trace_avg + (5 * (roi)), linewidth=0.5)
        except:
            print("no roi")
else:
    for roi in range(1, n_rois+1):
        curr_roi = np.where(rois == roi)
        #trace = aligned[:, i, curr_roi[1], curr_roi[2]]

        try:
            z = curr_roi[0][1]
            center_x = np.nanmean(curr_roi[2])
            center_y = np.nanmean(curr_roi[1])
            axs2[0].scatter(center_x, center_y, s=2)

            num_pix = curr_roi[1]
            trace = zscore(np.nanmean(aligned[:, z, curr_roi[1], curr_roi[2]], axis=1))
            trace_reshape = np.reshape(trace[:768*3], (3, np.shape(aligned)[0]//3))
            trace_avg = np.nanmean(trace_reshape, axis=0)

            axs2[1].plot(t_imaging[:768], trace_avg + (5 * (roi)), linewidth=0.5)
        except:
            print("no roi")

    

In [ ]:
fig.savefig(fish / "manual rois and traces.pdf", dpi=300)
fig.savefig(fish / "manual rois and traces.jpg", dpi=300)

In [ ]:
fig2.savefig(fish / "manual rois and traces avg.pdf", dpi=300)
fig2.savefig(fish / "manual rois and traces avg.jpg", dpi=300)